[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/099.%20The%20Electric%20Vehicle%20Routing%20Problem%20%28EVRP%29/P99-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 7: Human–AI Symbiotic Partnership

### Purpose and scope

This notebook demonstrates a human–AI collaboration layer for decisions that arise in electric-vehicle routing. An explainable AI component generates heuristic recommendations; a simulated human partner responds; and the partnership records feedback, trust measures, and collaboration indicators.

This is a simulation of the decision-support process, not an EVRP optimization model. It does not calculate feasible vehicle routes, enforce battery-feasibility or time-window constraints, or establish that a recommendation is operationally safe. Recommendation outcomes and baseline comparisons are illustrative simulation values, not measured field performance. The example therefore focuses on how recommendations and feedback are represented and analyzed, rather than on finding an optimal route.

### Inputs and outputs

The example uses three simulated decision scenarios, a human-partner profile, and fixed random seeds. It produces explainable recommendations, simulated feedback, session-level trust and collaboration measures, and visual summaries. The numeric scores are defined by the rules in the code; they should be interpreted as indicators within this demonstration, not validated measures of real-world performance.

### Partnership assumptions

- A human operator and an AI decision-support system contribute different information to decisions.
- Recommendations include a proposed action, reasoning, alternatives, confidence, and risk factors.
- Human feedback is represented by acceptance, modifications, reasoning, and decision confidence.
- Trust changes incrementally according to the feedback rules implemented below.
- The simulated feedback process is not a behavioral model of real operators.

### Workflow

1. Represent recommendations, feedback, trust, partners, and sessions as data structures.
2. Generate route-planning and charging recommendations from simple scenario rules.
3. Adapt the explanation to the human partner's stated preference.
4. Simulate human feedback and update trust and collaboration records.
5. repeat the session to examine how the indicators evolve under the simulation rules.
6. Visualize the indicators and inspect an explicitly illustrative comparison with baseline values.

### Example scenario

The surrounding EVRP context is represented by one depot, six customers, two charging stations, two electric vehicles, a nominal battery capacity of 100 kWh, and energy consumption of 0.8 kWh per kilometre. These values describe the broader problem setting; the simulation code below does not use them to construct or validate vehicle routes. Its decision scenarios instead provide battery percentages, pending deliveries, and nearby stations as inputs to recommendation rules.

### What to look for

- Whether each recommendation exposes its reasoning, alternatives, and risks.
- How simulated feedback affects trust and collaboration indicators.
- Which quantities are generated from scenario rules and which are random simulation values.
- Why illustrative comparisons cannot establish that one approach performs better in practice.


In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A fixed seed makes the simulated random draws reproducible when cells run in order.
np.random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")


class DecisionType(Enum):
    ROUTE_PLANNING = "route_planning"
    CHARGING_DECISION = "charging_decision"
    PRIORITY_ADJUSTMENT = "priority_adjustment"
    CONSTRAINT_HANDLING = "constraint_handling"
    EMERGENCY_RESPONSE = "emergency_response"


class ConfidenceLevel(Enum):
    VERY_LOW = 0.2
    LOW = 0.4
    MEDIUM = 0.6
    HIGH = 0.8
    VERY_HIGH = 1.0


@dataclass
class AIRecommendation:
    """A proposed action together with its explanation and uncertainty label."""

    recommendation_type: DecisionType
    primary_action: str
    confidence: ConfidenceLevel
    reasoning: List[str]
    alternatives: List[str]
    expected_outcome: Dict[str, float]
    risk_factors: List[str]
    data_sources: List[str]
    timestamp: datetime = field(default_factory=datetime.now)


@dataclass
class HumanFeedback:
    """A simulated human response to one recommendation."""

    recommendation_id: str
    acceptance_level: float
    modifications: List[str]
    reasoning: str
    confidence_in_decision: float
    additional_context: Dict[str, Any]
    timestamp: datetime = field(default_factory=datetime.now)


@dataclass
class TrustMetrics:
    """Trust indicators, each represented on a 0-to-1 scale."""

    overall_trust: float
    competence_trust: float
    integrity_trust: float
    benevolence_trust: float
    predictability_trust: float
    trust_history: List[float] = field(default_factory=list)


@dataclass
class HumanPartner:
    """Profile and preferences for the simulated human partner."""

    partner_id: int
    name: str
    expertise_level: float
    risk_tolerance: float
    decision_style: str
    preferred_explanation_detail: str
    trust_in_ai: TrustMetrics
    learning_rate: float = 0.1
    adaptation_history: List[Dict[str, Any]] = field(default_factory=list)


@dataclass
class CollaborationSession:
    """Records recommendations, feedback, and outcomes for one session."""

    session_id: str
    human_partner: HumanPartner
    start_time: datetime
    end_time: Optional[datetime]
    decisions_made: List[Dict[str, Any]]
    ai_recommendations: List[AIRecommendation]
    human_feedback: List[HumanFeedback]
    collaboration_score: float
    learning_outcomes: List[str]


## 1. Explainable recommendation engine

A recommendation is more than an action label: it also carries a confidence category, reasons, alternatives, risks, and named data sources. This structure makes the decision-support output inspectable. In this demonstration, the rules are hand-coded and the expected outcomes are sampled for simulation; they are not predictions from a trained or validated routing model.

The route and charging rules use thresholds and scenario information to select a recommendation. They do not solve an optimization problem. The `confidence` value is a discrete communication label, not a calibrated probability that the action is correct. The values under `expected_outcome` are random draws and should not be interpreted as verified estimates.


In [ ]:
class ExplainableAIEngine:
    """Generate rule-based recommendations with inspectable explanations."""

    def __init__(self):
        self.knowledge_base = {
            "routing_principles": [
                "Minimize total distance while respecting battery constraints",
                "Consider charging station availability and wait times",
                "Prioritize time-sensitive deliveries",
                "Balance workload across vehicles",
            ],
            "battery_management": [
                "Maintain 20% battery buffer for emergencies",
                "Charge during off-peak hours when possible",
                "Consider temperature effects on battery performance",
                "Plan charging stops to minimize total trip time",
            ],
            "operational_constraints": [
                "Respect driver working hours and breaks",
                "Consider customer preferred delivery windows",
                "Account for traffic patterns and road conditions",
                "Follow safety regulations and protocols",
            ],
        }
        self.decision_history = deque(maxlen=1000)

    def generate_route_recommendation(
        self,
        current_state: Dict[str, Any],
        human_preferences: Dict[str, Any],
    ) -> AIRecommendation:
        """Select a route-planning message using simple scenario thresholds."""
        battery_level = current_state.get("battery_level", 50.0)
        pending_deliveries = current_state.get("pending_deliveries", [])

        if battery_level < 30.0:
            primary_action = "Proceed to the nearest charging station immediately"
            confidence = ConfidenceLevel.VERY_HIGH
            reasoning = [
                f"Battery level is critically low at {battery_level:.1f}%",
                "Continuing without charging may leave insufficient energy",
                "A charging stop is the conservative action in this scenario",
            ]
            alternatives = [
                "Complete a very nearby urgent delivery first",
                "Request mobile charging if it is available",
            ]
        elif len(pending_deliveries) > 3:
            primary_action = "Plan an efficient delivery sequence with a charging stop"
            confidence = ConfidenceLevel.HIGH
            reasoning = [
                f"{len(pending_deliveries)} deliveries are pending",
                "A planned charging stop may support continued operations",
                "Actual route feasibility requires distance and energy calculations",
            ]
            alternatives = [
                "Split deliveries into separate vehicle routes",
                "Prioritize urgent deliveries",
                "Include an additional charging stop",
            ]
        else:
            primary_action = "Review the remaining deliveries and plan their sequence"
            confidence = ConfidenceLevel.MEDIUM
            reasoning = [
                f"Battery level is {battery_level:.1f}% in the supplied scenario",
                "The number of pending deliveries is limited in this rule-based case",
                "Route feasibility cannot be inferred from battery percentage alone",
            ]
            alternatives = [
                "Consider a preventive charging stop",
                "Reorder deliveries using operator priorities",
                "Check for nearby pickup opportunities",
            ]

        expected_outcome = {
            "total_distance": float(np.random.uniform(15, 25)),
            "energy_consumed": float(np.random.uniform(12, 20)),
            "completion_time": float(np.random.uniform(2, 4)),
            "success_probability": confidence.value,
        }
        risk_factors = []
        if battery_level < 40.0:
            risk_factors.append("Low battery level increases the risk of power depletion")
        if len(pending_deliveries) > 5:
            risk_factors.append("A high delivery volume may cause delays")
        if confidence == ConfidenceLevel.MEDIUM:
            risk_factors.append("The rule-based recommendation omits important route details")

        return AIRecommendation(
            recommendation_type=DecisionType.ROUTE_PLANNING,
            primary_action=primary_action,
            confidence=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            expected_outcome=expected_outcome,
            risk_factors=risk_factors,
            data_sources=[
                "Simulated battery state",
                "Scenario delivery list",
                "Scenario charging-station list",
            ],
        )

    def generate_charging_recommendation(
        self,
        current_state: Dict[str, Any],
        human_preferences: Dict[str, Any],
    ) -> AIRecommendation:
        """Select a charging message using battery and urgency thresholds."""
        battery_level = current_state.get("battery_level", 50.0)
        nearby_stations = current_state.get("nearby_stations", [])
        urgent_deliveries = current_state.get("urgent_deliveries", [])

        if battery_level < 25.0:
            primary_action = "Seek an available charging station promptly"
            confidence = ConfidenceLevel.VERY_HIGH
            reasoning = [
                f"Battery level is low at {battery_level:.1f}%",
                "Charging availability and travel distance must be checked before acting",
                "The scenario rule favors prompt attention to the battery state",
            ]
        elif battery_level < 45.0 and not urgent_deliveries:
            primary_action = "Consider charging during a low-demand period"
            confidence = ConfidenceLevel.HIGH
            reasoning = [
                f"Battery level is {battery_level:.1f}%",
                "No urgent deliveries are listed in this scenario",
                "Charging timing still depends on station access and route feasibility",
            ]
        else:
            primary_action = "Continue operations while monitoring the battery"
            confidence = ConfidenceLevel.MEDIUM
            reasoning = [
                f"Battery level is {battery_level:.1f}%",
                "The scenario rule does not identify an immediate charging trigger",
                "Operational feasibility should be checked against the planned route",
            ]

        alternatives = [
            "Check another charging station for availability and charging rate",
            "Coordinate charging with a scheduled break",
            "Check whether mobile charging is available",
        ]
        expected_outcome = {
            "charging_time": float(np.random.uniform(0.5, 2.0)),
            "cost": float(np.random.uniform(5, 15)),
            "battery_after_charging": float(
                min(100.0, battery_level + np.random.uniform(40, 60))
            ),
            "operational_impact": float(np.random.uniform(0.1, 0.3)),
        }
        risk_factors = []
        if not nearby_stations:
            risk_factors.append("No nearby charging stations are listed in the scenario")
        if urgent_deliveries:
            risk_factors.append("Charging may delay urgent deliveries")

        return AIRecommendation(
            recommendation_type=DecisionType.CHARGING_DECISION,
            primary_action=primary_action,
            confidence=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            expected_outcome=expected_outcome,
            risk_factors=risk_factors,
            data_sources=[
                "Simulated battery state",
                "Scenario charging-station list",
                "Scenario delivery schedule",
            ],
        )

    def adapt_to_human_feedback(self, feedback: HumanFeedback) -> None:
        """Record feedback for inspection; no model is trained in this example."""
        if feedback.acceptance_level < 0.5:
            self.decision_history.append(
                {
                    "type": "rejected_or_modified",
                    "reasoning": feedback.reasoning,
                    "modifications": feedback.modifications,
                    "timestamp": feedback.timestamp,
                }
            )
        elif feedback.acceptance_level > 0.8:
            self.decision_history.append(
                {
                    "type": "accepted",
                    "reasoning": feedback.reasoning,
                    "confidence": feedback.confidence_in_decision,
                    "timestamp": feedback.timestamp,
                }
            )


ai_engine = ExplainableAIEngine()
print(f"Created explainable recommendation engine with {len(ai_engine.knowledge_base)} knowledge domains")


## 2. Partnership, feedback, and trust updates

The partnership connects a human profile to the recommendation engine. Each feedback record contains an acceptance score in the interval $[0,1]$. The simulation updates overall trust by a small step proportional to the difference between acceptance and a neutral reference value of $0.5$:

$$
T_{k+1} = \operatorname{clip}\left(T_k + 0.1(a_k - 0.5), 0, 1\right),
$$

where $T_k$ is overall trust before feedback $k$, $a_k$ is the acceptance level, and `clip` limits the result to the interval $[0,1]$. Other trust components use similarly explicit heuristic updates in the code. These equations define the simulation's scoring rules; they are not empirical laws of human trust.

Confidence labels describe the recommendation engine's category. The partnership does not reinterpret them as statistically calibrated probabilities. Human preferences affect explanation presentation, while the recommendation rules themselves remain scenario-based.


In [ ]:
class HumanAIPartnership:
    """Manage recommendation presentation, simulated feedback, and session metrics."""

    def __init__(self, human_partner: HumanPartner, ai_engine: ExplainableAIEngine):
        self.human_partner = human_partner
        self.ai_engine = ai_engine
        self.current_session = None
        self.session_history = []
        self.collaboration_efficiency = 0.5
        self.decision_quality_score = 0.5
        self.trust_development_rate = 0.0
        self.shared_knowledge_base = defaultdict(list)
        self.communication_patterns = defaultdict(int)
        self.explanation_style = human_partner.preferred_explanation_detail
        self.decision_support_level = "collaborative"

    def start_collaboration_session(
        self, problem_context: Dict[str, Any]
    ) -> CollaborationSession:
        """Create a fresh record so each simulated session has its own feedback."""
        session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
        session = CollaborationSession(
            session_id=session_id,
            human_partner=self.human_partner,
            start_time=datetime.now(),
            end_time=None,
            decisions_made=[],
            ai_recommendations=[],
            human_feedback=[],
            collaboration_score=0.0,
            learning_outcomes=[],
        )
        self.current_session = session
        return session

    def generate_ai_recommendation(
        self, decision_type: DecisionType, current_state: Dict[str, Any]
    ) -> AIRecommendation:
        """Generate a recommendation and adapt its explanation to the partner."""
        human_preferences = {
            "risk_tolerance": self.human_partner.risk_tolerance,
            "expertise_level": self.human_partner.expertise_level,
            "decision_style": self.human_partner.decision_style,
            "explanation_detail": self.human_partner.preferred_explanation_detail,
        }

        if decision_type == DecisionType.ROUTE_PLANNING:
            recommendation = self.ai_engine.generate_route_recommendation(
                current_state, human_preferences
            )
        elif decision_type == DecisionType.CHARGING_DECISION:
            recommendation = self.ai_engine.generate_charging_recommendation(
                current_state, human_preferences
            )
        else:
            recommendation = AIRecommendation(
                recommendation_type=decision_type,
                primary_action="Consult the human partner before deciding",
                confidence=ConfidenceLevel.MEDIUM,
                reasoning=["This decision type is not covered by the scenario rules"],
                alternatives=["Proceed cautiously", "Gather additional information"],
                expected_outcome={"success_probability": 0.6},
                risk_factors=["The current scenario does not specify a decision rule"],
                data_sources=["Human expertise", "Scenario information"],
            )

        return self._adapt_explanation_to_human(recommendation)

    def _adapt_explanation_to_human(
        self, recommendation: AIRecommendation
    ) -> AIRecommendation:
        """Adjust explanation length without changing the recommended action."""
        detail = self.human_partner.preferred_explanation_detail
        if detail == "brief":
            recommendation.reasoning = recommendation.reasoning[:2]
            recommendation.alternatives = recommendation.alternatives[:1]
        elif detail == "detailed":
            recommendation.reasoning.extend(
                [
                    "The recommendation follows the stated scenario rules",
                    "Route feasibility requires operational data not modeled here",
                    "The listed alternatives support human review of the decision",
                ]
            )
        elif detail == "technical":
            recommendation.reasoning.extend(
                [
                    f"Confidence category value: {recommendation.confidence.value:.2f}",
                    "The confidence category is not a calibrated probability",
                    "Expected outcomes are simulated values, not validated forecasts",
                ]
            )
        return recommendation

    def process_human_feedback(
        self,
        recommendation: AIRecommendation,
        human_response: Dict[str, Any],
    ) -> HumanFeedback:
        """Validate, store, and use simulated feedback to update partnership metrics."""
        acceptance_level = float(human_response.get("acceptance_level", 0.5))
        confidence = float(human_response.get("confidence_in_decision", 0.5))
        if not 0.0 <= acceptance_level <= 1.0:
            raise ValueError("Acceptance level must be between 0 and 1.")
        if not 0.0 <= confidence <= 1.0:
            raise ValueError("Decision confidence must be between 0 and 1.")

        feedback = HumanFeedback(
            recommendation_id=f"rec_{recommendation.timestamp.strftime('%Y%m%d%H%M%S%f')}",
            acceptance_level=acceptance_level,
            modifications=list(human_response.get("modifications", [])),
            reasoning=str(human_response.get("reasoning", "")),
            confidence_in_decision=confidence,
            additional_context=dict(human_response.get("additional_context", {})),
        )
        self.ai_engine.adapt_to_human_feedback(feedback)
        self._update_trust_metrics(feedback)
        self._update_collaboration_patterns(feedback)
        if self.current_session is not None:
            self.current_session.human_feedback.append(feedback)
        return feedback

    def _update_trust_metrics(self, feedback: HumanFeedback) -> None:
        """Apply bounded heuristic updates to the trust indicators."""
        trust = self.human_partner.trust_in_ai
        trust.overall_trust = float(
            np.clip(trust.overall_trust + (feedback.acceptance_level - 0.5) * 0.1, 0, 1)
        )
        alignment = abs(feedback.acceptance_level - feedback.confidence_in_decision)
        trust.competence_trust = float(
            np.clip(trust.competence_trust + (1 - alignment) * 0.05, 0, 1)
        )
        integrity_change = 0.02 if len(feedback.reasoning) > 10 else -0.01
        trust.integrity_trust = float(
            np.clip(trust.integrity_trust + integrity_change, 0, 1)
        )
        trust.benevolence_trust = float(
            np.clip(
                trust.benevolence_trust
                + (feedback.confidence_in_decision - 0.5) * 0.05,
                0,
                1,
            )
        )
        if self.current_session and len(self.current_session.human_feedback) > 1:
            recent = self.current_session.human_feedback[-5:]
            variance = np.var([item.acceptance_level for item in recent])
            trust.predictability_trust = float(
                np.clip(trust.predictability_trust + (1 - variance) * 0.03, 0, 1)
            )
        trust.trust_history.append(trust.overall_trust)

    def _update_collaboration_patterns(self, feedback: HumanFeedback) -> None:
        """Summarize acceptance patterns and update the efficiency indicator."""
        if feedback.acceptance_level > 0.7:
            self.communication_patterns["high_acceptance"] += 1
        elif feedback.acceptance_level < 0.3:
            self.communication_patterns["low_acceptance"] += 1
        if feedback.modifications:
            self.communication_patterns["human_modifications"] += 1
        if self.current_session and self.current_session.human_feedback:
            recent_mean = np.mean(
                [item.acceptance_level for item in self.current_session.human_feedback]
            )
            self.collaboration_efficiency = float(
                0.7 * self.collaboration_efficiency + 0.3 * recent_mean
            )

    def calculate_collaboration_score(self) -> float:
        """Combine trust, efficiency, decision quality, and recorded learning."""
        if self.current_session is None:
            return 0.0
        session = self.current_session
        trust_score = self.human_partner.trust_in_ai.overall_trust
        efficiency_score = self.collaboration_efficiency
        if session.human_feedback:
            acceptance = np.mean([item.acceptance_level for item in session.human_feedback])
            confidence = np.mean(
                [item.confidence_in_decision for item in session.human_feedback]
            )
            decision_quality = (acceptance + confidence) / 2
        else:
            decision_quality = 0.5
        learning_score = min(len(session.learning_outcomes) / 10.0, 1.0)
        return float(
            0.30 * trust_score
            + 0.30 * efficiency_score
            + 0.25 * decision_quality
            + 0.15 * learning_score
        )


human_partner = HumanPartner(
    partner_id=1,
    name="Alex Chen",
    expertise_level=0.8,
    risk_tolerance=0.4,
    decision_style="analytical",
    preferred_explanation_detail="detailed",
    trust_in_ai=TrustMetrics(
        overall_trust=0.6,
        competence_trust=0.7,
        integrity_trust=0.8,
        benevolence_trust=0.6,
        predictability_trust=0.5,
    ),
)
partnership = HumanAIPartnership(human_partner, ai_engine)
print(f"Created human-AI partnership with {human_partner.name}")
print(f"Initial overall trust: {human_partner.trust_in_ai.overall_trust:.2f}")
print(f"Decision style: {human_partner.decision_style}")
print(f"Preferred explanation detail: {human_partner.preferred_explanation_detail}")


## 3. Simulate a collaboration session

The session contains three decision points: route planning, charging, and route planning. For each point, the engine creates a recommendation, a response is simulated, and the response is recorded. The simulation uses random draws for expected outcomes and human feedback; the fixed random seed makes those draws reproducible when the notebook is run from the beginning.

The feedback generator represents a deliberately simplified behavioral rule. It does not represent a real human participant, and the resulting acceptance values are not evidence about how operators generally respond to AI.

In [ ]:
def simulate_human_response(
    recommendation: AIRecommendation, human_partner: HumanPartner
) -> Dict[str, Any]:
    """Generate a reproducible, stochastic response for demonstration purposes."""
    base_acceptance = (
        human_partner.trust_in_ai.overall_trust + recommendation.confidence.value
    ) / 2
    if human_partner.decision_style == "analytical":
        base_acceptance += 0.2 if recommendation.confidence.value > 0.7 else -0.1
    elif human_partner.decision_style == "intuitive":
        base_acceptance -= 0.1

    acceptance = float(np.clip(np.random.normal(base_acceptance, 0.2), 0, 1))
    if acceptance > 0.7:
        reasoning = "The recommendation aligns with my analysis in this simulated response"
        modifications = []
        confidence = float(0.8 + np.random.uniform(0, 0.2))
        final_action = recommendation.primary_action
    elif acceptance > 0.4:
        reasoning = "The recommendation is reasonable, with some simulated modifications"
        modifications = ["Adjust timing", "Consider additional factors"]
        confidence = float(np.clip(0.6 + np.random.uniform(-0.1, 0.2), 0, 1))
        final_action = (
            recommendation.alternatives[0]
            if recommendation.alternatives
            else recommendation.primary_action
        )
    else:
        reasoning = "The simulated human response disagrees with the recommendation"
        modifications = ["Change approach", "Consider different priorities"]
        confidence = float(0.7 + np.random.uniform(0, 0.3))
        final_action = "Manual decision based on human judgment"

    return {
        "acceptance_level": acceptance,
        "modifications": modifications,
        "reasoning": reasoning,
        "confidence_in_decision": confidence,
        "additional_context": {
            "human_expertise_applied": human_partner.expertise_level,
            "simulated_time_pressure": float(np.random.uniform(0, 1)),
        },
        "final_action": final_action,
    }


def generate_learning_outcomes(session: CollaborationSession) -> List[str]:
    """Summarize simple patterns present in the session record."""
    outcomes = []
    if session.human_feedback:
        average_acceptance = np.mean(
            [item.acceptance_level for item in session.human_feedback]
        )
        if average_acceptance > 0.7:
            outcomes.append("High simulated alignment between recommendations and feedback")
        elif average_acceptance < 0.4:
            outcomes.append("The simulated recommendations received low acceptance")
        else:
            outcomes.append("Simulated acceptance was intermediate")

        average_confidence = np.mean(
            [item.confidence_in_decision for item in session.human_feedback]
        )
        if average_confidence > 0.7:
            outcomes.append("Simulated decision confidence was high on average")
        else:
            outcomes.append("Simulated decision confidence leaves room for review")

    if len(session.human_partner.trust_in_ai.trust_history) > 1:
        history = session.human_partner.trust_in_ai.trust_history
        trust_change = history[-1] - history[-2]
        if trust_change > 0.05:
            outcomes.append("Overall trust increased after the latest feedback")
        elif trust_change < -0.05:
            outcomes.append("Overall trust decreased after the latest feedback")
        else:
            outcomes.append("Overall trust changed only slightly after the latest feedback")
    return outcomes


def simulate_human_ai_collaboration(
    partnership: HumanAIPartnership,
) -> CollaborationSession:
    """Run the supplied scenarios using the specified partnership instance."""
    print("Starting Human-AI Collaboration Session")
    print("=" * 50)
    problem_context = {
        "problem_type": "ev_routing_decision_support",
        "urgency": "medium",
        "constraints": ["battery_limit", "time_windows", "charging_availability"],
    }
    session = partnership.start_collaboration_session(problem_context)
    print(f"Session {session.session_id} started at {session.start_time:%H:%M:%S}")

    scenarios = [
        {
            "type": DecisionType.ROUTE_PLANNING,
            "state": {
                "battery_level": 35.0,
                "location": (10, 15),
                "pending_deliveries": [
                    {"id": 1, "location": (20, 25), "deadline": "14:00", "priority": "high"},
                    {"id": 2, "location": (8, 30), "deadline": "15:30", "priority": "medium"},
                    {"id": 3, "location": (25, 10), "deadline": "16:00", "priority": "low"},
                ],
                "nearby_stations": [(12, 18), (22, 12)],
            },
        },
        {
            "type": DecisionType.CHARGING_DECISION,
            "state": {
                "battery_level": 28.0,
                "location": (20, 25),
                "urgent_deliveries": [{"id": 1, "deadline": "13:30"}],
                "nearby_stations": [(22, 22), (18, 28)],
            },
        },
        {
            "type": DecisionType.ROUTE_PLANNING,
            "state": {
                "battery_level": 75.0,
                "location": (15, 20),
                "pending_deliveries": [
                    {"id": 4, "location": (30, 15), "deadline": "17:00", "priority": "medium"},
                    {"id": 5, "location": (10, 8), "deadline": "18:00", "priority": "low"},
                ],
                "nearby_stations": [(18, 18), (25, 25)],
            },
        },
    ]

    for index, scenario in enumerate(scenarios, start=1):
        decision_type = scenario["type"]
        print(f"\nDecision point {index}: {decision_type.value}")
        print("-" * 35)
        recommendation = partnership.generate_ai_recommendation(
            decision_type, scenario["state"]
        )
        print(f"Recommendation confidence category: {recommendation.confidence.value:.2f}")
        print(f"Primary action: {recommendation.primary_action}")
        print("Reasoning:")
        for item_number, reason in enumerate(recommendation.reasoning, start=1):
            print(f"  {item_number}. {reason}")
        if recommendation.alternatives:
            print("Alternatives:")
            for item_number, alternative in enumerate(recommendation.alternatives, start=1):
                print(f"  {item_number}. {alternative}")
        if recommendation.risk_factors:
            print("Risk factors:")
            for item_number, risk in enumerate(recommendation.risk_factors, start=1):
                print(f"  {item_number}. {risk}")
        print("Simulated expected outcomes:")
        for key, value in recommendation.expected_outcome.items():
            print(f"  {key}: {value:.2f}")

        response = simulate_human_response(recommendation, partnership.human_partner)
        print(f"Simulated acceptance: {response['acceptance_level']:.2f}")
        print(f"Response reasoning: {response['reasoning']}")
        print(f"Decision confidence: {response['confidence_in_decision']:.2f}")
        if response["modifications"]:
            print(f"Modifications: {', '.join(response['modifications'])}")
        partnership.process_human_feedback(recommendation, response)
        session.ai_recommendations.append(recommendation)
        session.decisions_made.append(
            {
                "scenario": scenario,
                "recommendation": recommendation,
                "human_response": response,
                "final_decision": response.get("final_action", recommendation.primary_action),
            }
        )

    session.end_time = datetime.now()
    session.learning_outcomes = generate_learning_outcomes(session)
    session.collaboration_score = partnership.calculate_collaboration_score()
    partnership.session_history.append(session)
    print(f"\nSession completed at {session.end_time:%H:%M:%S}")
    print(f"Illustrative collaboration score: {session.collaboration_score:.2f}")
    print(f"Recorded learning indicators: {len(session.learning_outcomes)}")
    return session


session = simulate_human_ai_collaboration(partnership)


## 4. Partnership evolution across sessions

Each repetition uses the same scenario structure but draws new simulated values from the seeded random generator. Trust and collaboration state persist through the same partnership object, while each session receives a separate record. The trend is a descriptive linear slope through the observed session values; it is not a causal estimate and does not establish that a real partnership would improve over time.


In [ ]:
def analyze_partnership_evolution(
    partnership: HumanAIPartnership, num_sessions: int = 5
) -> Dict[str, Any]:
    """Run sessions and summarize their simulated partnership indicators."""
    if num_sessions < 1:
        raise ValueError("At least one session is required for the analysis.")

    print("Analyzing partnership evolution in the simulation")
    print("=" * 55)
    trust_evolution = []
    collaboration_scores = []
    acceptance_rates = []

    for session_number in range(num_sessions):
        print(f"\nSession {session_number + 1} of {num_sessions}")
        current_session = simulate_human_ai_collaboration(partnership)
        trust_evolution.append(partnership.human_partner.trust_in_ai.overall_trust)
        collaboration_scores.append(current_session.collaboration_score)
        acceptance_rates.append(
            float(np.mean([item.acceptance_level for item in current_session.human_feedback]))
        )
        print(f"Overall trust indicator: {trust_evolution[-1]:.3f}")
        print(f"Collaboration score: {collaboration_scores[-1]:.3f}")
        print(f"Mean simulated acceptance: {acceptance_rates[-1]:.3f}")

        if session_number > 2:
            trust = partnership.human_partner.trust_in_ai
            trust.overall_trust = float(np.clip(trust.overall_trust + 0.02, 0, 1))
            partnership.human_partner.expertise_level = min(
                1.0, partnership.human_partner.expertise_level + 0.01
            )

    trust_trend = float(np.polyfit(range(num_sessions), trust_evolution, 1)[0])
    collaboration_trend = float(
        np.polyfit(range(num_sessions), collaboration_scores, 1)[0]
    ) if num_sessions > 1 else 0.0
    trust = partnership.human_partner.trust_in_ai
    trust_components = {
        "Competence": trust.competence_trust,
        "Integrity": trust.integrity_trust,
        "Benevolence": trust.benevolence_trust,
        "Predictability": trust.predictability_trust,
    }

    print("\nSummary of simulated indicators")
    print(f"Trust trend slope per session: {trust_trend:+.4f}")
    print(f"Collaboration trend slope per session: {collaboration_trend:+.4f}")
    print(f"Mean collaboration score: {np.mean(collaboration_scores):.3f}")
    print(f"Mean acceptance: {np.mean(acceptance_rates):.3f}")
    print("Trust-component indicators:")
    for component, value in trust_components.items():
        print(f"  {component}: {value:.3f}")

    return {
        "trust_evolution": trust_evolution,
        "collaboration_scores": collaboration_scores,
        "acceptance_rates": acceptance_rates,
        "trust_trend": trust_trend,
        "collab_trend": collaboration_trend,
        "trust_components": trust_components,
    }


evolution_analysis = analyze_partnership_evolution(partnership, num_sessions=5)


## 5. Visualize the indicators

The panels show the recorded trust indicator, collaboration score, acceptance rate, and trust-component values. The trend line is descriptive. All plotted values come from the simulation; the chart does not measure operational routing quality or validate the trust model.


In [ ]:
def visualize_human_ai_partnership(
    partnership: HumanAIPartnership, evolution_analysis: Dict[str, Any]
) -> None:
    """Plot simulated partnership indicators with labeled axes and legends."""
    trust_values = evolution_analysis["trust_evolution"]
    collaboration = evolution_analysis["collaboration_scores"]
    acceptance = evolution_analysis["acceptance_rates"]
    sessions = np.arange(1, len(trust_values) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Simulated Human–AI Partnership Indicators", fontsize=15, weight="bold")

    axes[0, 0].plot(sessions, trust_values, marker="o", label="Overall trust indicator")
    if len(sessions) > 1:
        slope, intercept = np.polyfit(sessions, trust_values, 1)
        axes[0, 0].plot(
            sessions,
            slope * sessions + intercept,
            linestyle="--",
            label=f"Descriptive linear trend (slope {slope:.3f})",
        )
    axes[0, 0].set(title="Trust indicator by session", xlabel="Session", ylabel="Score (0–1)")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(sessions, collaboration, marker="s", label="Collaboration score")
    axes[0, 1].plot(sessions, acceptance, marker="^", label="Mean simulated acceptance")
    axes[0, 1].set(title="Session indicators", xlabel="Session", ylabel="Score (0–1)")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    components = evolution_analysis["trust_components"]
    bars = axes[1, 0].bar(list(components), list(components.values()))
    axes[1, 0].set(title="Trust-component indicators", ylabel="Score (0–1)")
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].tick_params(axis="x", rotation=20)
    axes[1, 0].grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, components.values()):
        axes[1, 0].text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.02,
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

    axes[1, 1].axis("off")
    partner = partnership.human_partner
    summary = (
        "Simulation context\n"
        "------------------\n"
        f"Partner profile: {partner.name}\n"
        f"Expertise parameter: {partner.expertise_level:.2f}\n"
        f"Decision style: {partner.decision_style}\n"
        f"Risk-tolerance parameter: {partner.risk_tolerance:.2f}\n\n"
        "Interpretation notes\n"
        "--------------------\n"
        "Scores follow hand-coded update rules.\n"
        "Acceptance is generated by a stochastic simulation.\n"
        "No route-feasibility or service-quality metric is calculated.\n"
        f"Sessions represented: {len(sessions)}"
    )
    axes[1, 1].text(
        0.04,
        0.96,
        summary,
        transform=axes[1, 1].transAxes,
        va="top",
        family="monospace",
        bbox={"boxstyle": "round", "facecolor": "aliceblue", "alpha": 0.8},
    )

    fig.tight_layout()
    plt.show()


visualize_human_ai_partnership(partnership, evolution_analysis)


## 6. Illustrative comparison with baseline values

The following table retains a small set of supplied example values to demonstrate how metric direction affects a comparison. These values are assumptions for illustration; they are not generated by the session simulation, measured from a study, or supported by an experimental design. A percentage change is therefore arithmetic on the stated values, not evidence of a performance advantage.

For decision time and error rate, lower values are preferable. For decision quality, adaptability, satisfaction, and learning rate, higher values are preferable. The output reports a direction-aware percentage change relative to each stated baseline; a negative value indicates movement in the undesirable direction for that metric.


In [ ]:
def compare_with_baseline_approaches() -> Dict[str, Dict[str, float]]:
    """Compare illustrative assumed values; these are not empirical results."""
    approaches = {
        "Human–AI Partnership": {
            "decision_time": 2.5,
            "decision_quality": 0.85,
            "error_rate": 0.05,
            "adaptability": 0.90,
            "human_satisfaction": 0.88,
            "learning_rate": 0.15,
        },
        "Fully Manual": {
            "decision_time": 5.0,
            "decision_quality": 0.70,
            "error_rate": 0.12,
            "adaptability": 0.60,
            "human_satisfaction": 0.75,
            "learning_rate": 0.05,
        },
        "Fully Automated": {
            "decision_time": 0.5,
            "decision_quality": 0.80,
            "error_rate": 0.08,
            "adaptability": 0.40,
            "human_satisfaction": 0.60,
            "learning_rate": 0.02,
        },
        "Traditional AI Support": {
            "decision_time": 1.5,
            "decision_quality": 0.75,
            "error_rate": 0.10,
            "adaptability": 0.30,
            "human_satisfaction": 0.65,
            "learning_rate": 0.03,
        },
    }
    lower_is_better = {"decision_time", "error_rate"}

    print("Illustrative comparison of assumed values (not empirical results)")
    print("=" * 75)
    header = f"{'Approach':<25}" + "".join(f"{name:>19}" for name in next(iter(approaches.values())))
    print(header)
    for name, values in approaches.items():
        row = f"{name:<25}" + "".join(f"{value:>19.2f}" for value in values.values())
        print(row)

    partnership_values = approaches["Human–AI Partnership"]
    for baseline_name in (
        "Fully Manual",
        "Fully Automated",
        "Traditional AI Support",
    ):
        baseline = approaches[baseline_name]
        print(f"\nDirection-aware percentage change relative to {baseline_name}:")
        for metric, partnership_value in partnership_values.items():
            baseline_value = baseline[metric]
            if baseline_value == 0:
                print(f"  {metric}: undefined because the baseline value is zero")
                continue
            if metric in lower_is_better:
                change = (baseline_value - partnership_value) / baseline_value * 100
            else:
                change = (partnership_value - baseline_value) / baseline_value * 100
            print(f"  {metric}: {change:+.1f}% (positive is directionally favorable)")

    print(
        "\nInterpretation: these assumed values illustrate trade-offs only. "
        "They do not establish that any approach is superior in practice."
    )
    return approaches


comparison_results = compare_with_baseline_approaches()


## 7. Interpretation and limitations

### What the demonstration shows

- Recommendations can be represented together with reasons, alternatives, risk factors, and scenario data sources.
- Human feedback can be recorded and used to update explicitly defined trust and collaboration indicators.
- Explanation length can be adapted to a stated preference without changing the recommended action.
- Repeated simulated sessions make it possible to inspect how stateful indicators evolve under the chosen update rules.

### What it does not show

- It does not solve the electric vehicle routing problem or prove route feasibility.
- It does not enforce vehicle capacity, time windows, station availability, charging duration, or battery-energy constraints.
- Its confidence categories and trust indicators are not calibrated or empirically validated.
- Its expected outcomes and simulated human responses are not operational forecasts or observations of real people.
- The illustrative baseline values do not support causal or statistical claims about comparative performance.

The central idea of this tier is collaborative decision support: AI can organize scenario information and make its recommendations inspectable, while a human can review context and retain decision authority. In a deployed routing system, this collaboration layer would need to be connected to a validated routing model and operational data; those components are outside this notebook's scope.
